<a href="https://colab.research.google.com/github/OnwutaKelvin/ML-notebooks/blob/main/CIFAR10_CNN_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Trainig a CIFAR dataset on CNN model with major upgrade consisting Batch Norm and colour augmentation. There will also be a proper training/test/inference pipline



In [ ]:
#importing dependencies
!pip install torch torchvision matplotlib tqdm pillow

In [ ]:
#training
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

#Device
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)
print(device)

#Data Augumentation
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),

    #color augmentation
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1
    ),

    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2023, 0.1994, 0.2010]
    )
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2023, 0.1994, 0.2010]
    )
])

#Dataset
train_dataset = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=train_transform
)

test_dataset = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=test_transform
)

#Train/Validation split
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size

train_dataset, val_dataset = random_split(
    train_dataset,
    [train_size, val_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

cuda


100%|██████████| 170M/170M [00:14<00:00, 12.1MB/s]


In [ ]:
#CNN architecture with BatchNorm
import torch.nn as nn

class CIFAR10CNN(nn.Module):
  def __init__(self):
    super().__init__()

    self.features = nn.Sequential(

        #Block1
        nn.Conv2d(3, 32, kernel_size=3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(2),

        #Block2
        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2),

        #Block3
        nn.Conv2d(64, 128, kernel_size=3, padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(2),
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(128 * 4 * 4, 256),
        nn.ReLU(),
        nn.Dropout(0.3),

        nn.Linear(256, 10)
    )

  def forward(self, x):
    x = self.features(x)
    x = self.classifier(x)
    return(x)

In [ ]:
#Training Loop
from tqdm import tqdm
import torch.nn as nn

model = CIFAR10CNN().to(device)

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 20
best_val_acc = 0

for epoch in range(epochs):
  model.train()

  running_loss = 0
  correct = 0
  total = 0

  loop = tqdm(train_loader)

  for images, labels in loop:
    images = images.to(device)
    labels = labels.to(device)

    outputs = model(images)

    loss = loss_fn(outputs, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    running_loss += loss.item()

    _, predicted = torch.max(outputs, 1)

    total += labels.size(0)
    correct += (predicted == labels).sum().item()

    loop.set_description(
        f"Epochs: [{epoch+1}/{epochs}]"
    )

  train_acc = 100 * correct / total


  #VALIDATION LOOP
  model.eval()

  val_correct = 0
  val_total = 0

  with torch.no_grad():
    for images, labels in val_loader:
      images = images.to(device)
      labels = labels.to(device)

      outputs = model(images)

      _, predicted = torch.max(outputs, 1)

      val_total += labels.size(0)
      val_correct += (predicted == labels).sum().item()

val_acc = 100 * val_correct / val_total

print(
    f"Loss: {running_loss:.4f} | "
    f"Train Acc: {train_acc:.2f}% | "
    f"Val Acc: {val_acc:.2f}%"
)

#Save the model
if val_acc > best_val_acc:
  best_val_acc = val_acc

  torch.save(
      model.state_dict(),
      "best_model.pth"
  )
  print("Best Model Saved!")

Epochs: [20/20]: 100%|██████████| 625/625 [00:40<00:00, 15.28it/s]


Loss: 418.4142 | Train Acc: 77.05% | Val Acc: 75.00%
Best Model Saved!


In [ ]:
#Test Accuracy
model.load_state_dict(
    torch.load("best_model.pth")
)

model.eval()

correct = 0
total = 0

with torch.no_grad():
  for images, labels in test_loader:
    images = images.to(device)
    labels = labels.to(device)

    outputs = model(images)
    _, predicted = torch.max(outputs, 1)

    total += labels.size(0)
    correct += (predicted == labels).sum().item()

test_acc = 100 * correct / total

print(f"Test Accuracy: {test_acc:.2f}%")

Test Accuracy: 79.41%


In [ ]:
#Inference
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms
import numpy as np

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

classes = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck"
]

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2023, 0.1994, 0.2010]
    )
])

model = CIFAR10CNN().to(device)

model.load_state_dict(
    torch.load("best_model.pth")
)

model.eval()

image_path = "dog.jpg"
# Create a dummy image if 'dog.jpg' does not exist
try:
    image = Image.open(image_path).convert("RGB")
except FileNotFoundError:
    print(f"Warning: '{image_path}' not found. Creating a dummy image for demonstration.")
    # Create a blank 32x32 RGB image for demonstration
    dummy_image_data = np.random.randint(0, 256, (32, 32, 3), dtype=np.uint8)
    image = Image.fromarray(dummy_image_data, 'RGB')
    # Optionally save it if you want to see it, but not strictly necessary for this demo
    image.save(image_path)

image = transform(image)

image = image.unsqueeze(0).to(device)

with torch.no_grad():
  logits = model(image)

  probs = F.softmax(logits, dim=1)

  confidence, predicted = torch.max(
          probs,
          dim=1
      )
print(
    f"Prediction: "
    f"{classes[predicted.item()]}"
)

print(
    f"Confidence: "
    f"{confidence.item() * 100:.2f}%"
)

Prediction: frog
Confidence: 36.41%


/tmp/ipykernel_4352/1232488157.py:52: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(dummy_image_data, 'RGB')
